In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [8]:
# load datasets from games_clean and teams
clean_games = pd.read_csv('games_clean.csv')
teams = pd.read_csv('teams.csv')
X_train = pd.read_csv('train-test/X_train.csv')
y_train = pd.read_csv('train-test/y_train.csv')
X_test = pd.read_csv('train-test/X_test.csv')
y_test = pd.read_csv('train-test/y_test.csv')

In [9]:
clean_games

,GAME_ID,GAME_DATE_EST,SEASON,NICKNAME_home,NICKNAME_away,TEAM_ID_home,TEAM_ID_away,PTS_roll10_home,FG_PCT_roll10_home,FT_PCT_roll10_home,FG3_PCT_roll10_home,AST_roll10_home,REB_roll10_home,PTS_roll10_away,FG_PCT_roll10_away,FT_PCT_roll10_away,FG3_PCT_roll10_away,AST_roll10_away,REB_roll10_away,POINT_DIFF
0,20300127,2003-11-15,2003,Wizards,Spurs,1610612764,1610612759,92.8,0.4395,0.7134,0.3308,20.6,44.6,86.4,0.3806,0.7503,0.2891,18.3,46.8,-24.0
1,20300128,2003-11-15,2003,Hawks,Nets,1610612737,1610612751,89.4,0.4327,0.7390,0.3765,20.1,44.0,86.0,0.4149,0.7358,0.3413,23.9,40.2,-15.0
2,20300130,2003-11-15,2003,Cavaliers,76ers,1610612739,1610612755,89.5,0.4405,0.6874,0.3291,23.3,45.7,93.2,0.4382,0.7238,0.3170,19.6,38.3,3.0
3,20300136,2003-11-16,2003,Raptors,Rockets,1610612761,1610612745,75.3,0.3821,0.7709,0.2969,17.8,40.8,87.7,0.4390,0.7576,0.3872,17.4,41.7,4.0
4,20300139,2003-11-17,2003,76ers,Rockets,1610612755,1610612745,94.1,0.4390,0.7082,0.3209,20.5,38.9,89.4,0.4289,0.7770,0.3789,17.5,42.4,-8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26556,22200474,2022-12-21,2022,Thunder,Trail Blazers,1610612760,1610612757,114.9,0.4619,0.8106,0.3594,24.3,45.4,118.4,0.4992,0.7934,0.3991,24.7,43.9,3.0
26557,22200475,2022-12-21,2022,Kings,Lakers,1610612758,1610612747,115.6,0.4783,0.7720,0.3434,25.2,44.2,119.1,0.4932,0.7803,0.3441,25.2,43.8,14.0
26558,22200476,2022-12-21,2022,Clippers,Hornets,1610612746,1610612766,107.1,0.4216,0.7641,0.3739,21.2,47.3,114.1,0.4428,0.7642,0.3311,23.0,46.7,21.0
26559,22200477,2022-12-22,2022,Pelicans,Spurs,1610612740,1610612759,118.7,0.4896,0.7884,0.3460,26.7,45.0,112.5,0.4789,0.7387,0.3508,26.1,43.4,9.0


In [ ]:
# Naive baseline
naive_predictions = np.full(len(y_test), y_train.mean())
naive_rmse = np.sqrt(np.mean((naive_predictions - y_test.values.flatten()) ** 2))
naive_accuracy = np.mean((naive_predictions > 0.5) == (y_test.values.flatten() > 0.5))

print(f"Naive Baseline RMSE: {naive_rmse:.4f}")
print(f"Naive Baseline Accuracy: {naive_accuracy:.4f}")

Naive Baseline RMSE: 15.4116
Naive Baseline Accuracy: 0.5224


In [ ]:
# OLS baseline

# add a bias term to the features
X_train_ols = np.hstack([X_train.values, np.ones((X_train.shape[0], 1))])
X_test_ols = np.hstack([X_test.values, np.ones((X_test.shape[0], 1))])

beta_ols = np.linalg.lstsq(X_train_ols, y_train.values.flatten(), rcond=None)[0]

# make predictions and evaluate
ols_predictions = X_test_ols @ beta_ols
ols_rmse = np.sqrt(np.mean((ols_predictions - y_test.values.flatten()) ** 2))
ols_accuracy = np.mean((ols_predictions > 0.5) == (y_test.values.flatten() > 0.5))

print(f"OLS Baseline RMSE: {ols_rmse:.4f}")
print(f"OLS Baseline Accuracy: {ols_accuracy:.4f}")

OLS Baseline RMSE: 15.5505
OLS Baseline Accuracy: 0.5213


In [ ]:
layer_dimensions = [70, 64, 32, 1]

In [26]:
# MLP Implementation
np.random.seed(42)
eta = 0.01 # learning rate
num_epochs = 1000

# Initializing weights from normal distribution
# Using He initialization to allow variance to stay roughly constant between layers
W1 = np.random.normal(size=(70, 64)) * np.sqrt(2.0/70)
W2 = np.random.normal(size=(64, 32)) * np.sqrt(2.0/64)
W3 = np.random.normal(size=(32, 1)) * np.sqrt(2.0/32)

# Bias terms for each layer
b1 = np.zeros((64, 1))
b2 = np.zeros((32, 1))
b3 = np.zeros((1, 1))

In [27]:
# ReLU activation function - used for all hidden layers
def relu(X):
    return np.maximum(X, 0)

def relu_derivative(X):
    return np.where(X > 0)

In [ ]:
# Forward pass
def forward(X, W1, b1, W2, b2, W3, b3):
    z1 = np.dot(W1, X) + b1
    h1 = relu(z1)
    z2 = np.dot(W2, h1) + b2
    h2 = relu(z2)
    z3 = np.dot(W3, h2) + b3 # fw_x
    return z1, h1, z2, h2, z3.flatten()

In [29]:
# Loss function: half MSE
# Using half MSE instead of full MSE since it cancels out the 2 in the derivative of MSE

def half_mse_loss(y, fw_x):
    fw_x_flat = fw_x.flatten()
    y_flat = y.flatten()
    return 0.5/len(fw_x) * np.sum((fw_x_flat - y_flat) ** 2)

In [ ]:
# Backpropagation
def dLdW1(X, y, W1, W2, W3, h1, h2, z1, z2, fw_x):
    pass

def dLdW2(X, y, W1, W2, W3, h1, h2, z1, z2, fw_x):
    pass

def dLdW3(X, y, W1, W2, W3, h1, h2, z1, z2, fw_x):
    n = X.shape[0]
    dL_dz3 = (fw_x - y) / n
    dL_dw3 = h2 * dL_dz3.T
    dL_db3 = dL_dz3.sum(keepdims=True).reshape(1,1)

    return dL_dz3, dL_dw3, dL_db3

In [ ]:
# Training loop

# Tracking losses to ensure they converge
losses = []

def train_mlp(X_train, y_train, W1, W2, W3, b1, b2, b3):
    for epoch in range(num_epochs):
        # forward pass
        z1, h1, z2, h2, fw_x = forward(X_train, W1, b1, W2, b2, W3, b3)

        # find loss after forward pass
        loss = half_mse_loss(y=y_train, fw_x=fw_x)
        losses.append(loss)

        # backpropagation
        grad_W1 = dLdW1(X_train, y_train, W1, W2, W3, h1, h2, z1, z2, fw_x)
        grad_W2 = dLdW2(X_train, y_train, W1, W2, W3, h1, h2, z1, z2, fw_x)
        grad_W3 = dLdW3(X_train, y_train, W1, W2, W3, h1, h2, z1, z2, fw_x)

        W1 = W1 - eta * grad_W1
        W2 = W2 - eta * grad_W2
        W3 = W3 - eta * grad_W3

    return W1, W2, W3, b1, b2, b3